### Feature Extraction

### import libraries

In [2]:
%pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 3.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 2.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding
from gensim.models import Word2Vec

import warnings
warnings.filterwarnings("ignore")


### load the datasets and tokenizer

In [4]:
X_train_padded = np.load("../models/X_train_padded.npy")
X_val_padded = np.load("../models/X_val_padded.npy")
X_test_padded = np.load("../models/X_test_padded.npy")

y_train = np.load("../models/y_train.npy")
y_val = np.load("../models/y_val.npy")
y_test = np.load("../models/y_test.npy")

In [5]:
with open("../models/tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

In [6]:
print("Training:", X_train_padded.shape)
print("Validation:", X_val_padded.shape)
print("Testing:", X_test_padded.shape)

Training: (34705, 200)
Validation: (7439, 200)
Testing: (7438, 200)


#### Tokenization + Integer Encoding

In [7]:
word_index = tokenizer.word_index
print("Vocabulary size:", len(word_index))

Vocabulary size: 85869


In [8]:
list(word_index.items())[:20]

[('<OOV>', 1),
 ('movie', 2),
 ('film', 3),
 ('one', 4),
 ('like', 5),
 ('good', 6),
 ('time', 7),
 ('even', 8),
 ('would', 9),
 ('story', 10),
 ('really', 11),
 ('see', 12),
 ('well', 13),
 ('much', 14),
 ('bad', 15),
 ('get', 16),
 ('great', 17),
 ('people', 18),
 ('also', 19),
 ('first', 20)]

In [9]:
print(X_train_padded[0])

[   14    49     4   430   128   142   289     1   823    25    94    16
    29    62     4  7911  1116  3148   204   191  4360  3438  1178  1075
  2165   741 18543    32    14   472     1  1621     1   439  1107   185
  1613    19  2386  4360     1   202   573     1     1 14455     1  1028
 13716   105   550  1116     1   632 12225     1 12454  7288  4360  9447
   349  4360  5736  3340     1   201   812  2419   929  4579   340  2754
     1  3518     1  2025   211   684   991  3683  2165    64  1483     1
     1   714     1  1474  1116  2042   980    19   452  2719   204  1675
  7816    48    39   414   118  1588 11467    87  1500   185   968   894
  1291  6251  2452  6251  1958  2285    60     1     9    14    45  1879
   243   372   633   140     1     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0   

#### TF-IDF Representation

In [10]:
X_train_text = pd.read_pickle("../models/X_train_text.pkl")
X_val_text = pd.read_pickle("../models/X_val_text.pkl")
X_test_text = pd.read_pickle("../models/X_test_text.pkl")

In [11]:
print(X_train_text[0])

one reviewers mentioned watching oz episode hooked right exactly happened first thing struck oz brutality unflinching scenes violence set right word go trust show faint hearted timid show pulls punches regards drugs sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focuses mainly emerald city experimental section prison cells glass fronts face inwards privacy high agenda em city home many aryans muslims gangstas latinos christians italians irish scuffles death stares dodgy dealings shady agreements never far away would say main appeal show due fact goes shows dare forget pretty pictures painted mainstream audiences forget charm forget romance oz mess around first episode ever saw struck nasty surreal say ready watched developed taste oz got accustomed high levels graphic violence violence injustice crooked guards sold nickel inmates kill order get away well mannered middle class inmates turned prison bitches due lack street skills

In [12]:
tfidf = TfidfVectorizer(max_features=20000)
#only fit on training data to avoid data leakage
X_train_tfidf = tfidf.fit_transform(X_train_text)
#Transform validation data
X_val_tfidf = tfidf.transform(X_val_text)
#Transform test data
X_test_tfidf = tfidf.transform(X_test_text)

In [13]:
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (34705, 20000)
Validation TF-IDF shape: (7439, 20000)
Testing TF-IDF shape: (7438, 20000)


### Inspect TF-IDF Features

In [14]:
tfidf_features = tfidf.get_feature_names_out()
print("Number of TF-IDF features:", len(tfidf_features))
print("First 20 TF-IDF features:", tfidf_features[:20])

Number of TF-IDF features: 20000
First 20 TF-IDF features: ['aaliyah' 'aamir' 'aardman' 'aaron' 'ab' 'abandon' 'abandoned'
 'abandoning' 'abandonment' 'abandons' 'abba' 'abbey' 'abbie' 'abbot'
 'abbott' 'abby' 'abc' 'abducted' 'abduction' 'abe']


In [15]:
print(X_train_tfidf[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 94 stored elements and shape (1, 20000)>
  Coords	Values
  (0, 11736)	0.1080710323881538
  (0, 15455)	0.043715244960743686
  (0, 12391)	0.04949423359839865
  (0, 6168)	0.06655885041317369
  (0, 13749)	0.05261311118866898
  (0, 19796)	0.05447199953391698
  (0, 5544)	0.06201969734160988
  (0, 19987)	0.08674479539001421
  (0, 11721)	0.04018816870202602
  (0, 11073)	0.051270223792300125
  (0, 7422)	0.03675246043922442
  (0, 19421)	0.04005519779543977
  (0, 705)	0.046768493214350323
  (0, 14887)	0.1209194959703686
  (0, 19988)	0.2745218889887536
  (0, 11603)	0.09902460054688192
  (0, 16903)	0.11887378681083834
  (0, 9612)	0.06003311454936211
  (0, 2635)	0.4439928123857587
  (0, 9242)	0.10057055021612389
  (0, 2304)	0.08271117309400663
  (0, 894)	0.08302392800717329
  (0, 15796)	0.18691197769127804
  (0, 17966)	0.0741957576718199
  (0, 14526)	0.13868776928466112
  :	:
  (0, 4004)	0.10085972832274463
  (0, 1903)	0.09041333431884896

### Word Embeddings

In [16]:
sentences = [
    review.split() 
    for review in X_train_text
]

In [17]:
word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [18]:
word2vec_model.wv["movie"]

array([-1.260853  ,  0.3982071 ,  0.38670862, -0.3697563 , -1.2829472 ,
       -1.2267121 ,  0.5652774 ,  0.7146066 , -1.2335654 ,  1.1772237 ,
        1.361458  , -1.0646825 , -0.68370104,  0.34716758,  0.9178334 ,
        0.26142564,  0.9279897 , -0.41220832, -1.8377379 , -1.3107661 ,
       -0.8256027 ,  0.70896363,  0.4678066 , -0.29896554, -0.92300445,
        0.06292557, -0.82423437,  0.05747796, -0.57410455, -1.0351005 ,
        2.1266139 ,  0.7871471 ,  1.9443395 , -2.2016182 ,  0.03904045,
        0.5080582 ,  2.2888865 ,  0.07080251,  0.5169739 ,  0.7925288 ,
       -0.44598675,  1.3381997 , -0.41130674, -0.16806616,  0.0166133 ,
       -0.08308853, -0.63851815,  0.38771662,  1.1642778 , -0.79951745,
        0.49203786,  0.18982585,  2.4217794 ,  0.27414066,  1.3965471 ,
        1.2430784 ,  2.0459676 , -0.3921992 , -0.35095763,  2.3103714 ,
        0.2747686 , -0.96422   ,  1.7354513 , -0.48838544, -2.2380056 ,
       -0.9620365 ,  0.88777775,  0.13034064,  0.9134305 ,  1.74

In [19]:
print(word2vec_model.wv["movie"].shape)

(100,)


### Check Word Similarity

In [20]:
word2vec_model.wv.most_similar("movie", topn=10)

[('film', 0.7857902646064758),
 ('flick', 0.6738654375076294),
 ('movies', 0.6442257165908813),
 ('thats', 0.5758469700813293),
 ('think', 0.5612714290618896),
 ('sequel', 0.5591050982475281),
 ('guess', 0.5576024055480957),
 ('suppose', 0.5537834167480469),
 ('disappointed', 0.5533927083015442),
 ('opinion', 0.5460073947906494)]

### Trainable Embedding Layer

In [21]:
vocab_size = min(20000,len(tokenizer.word_index) + 1)
embedding_dim = 128
max_length = X_train_padded.shape[1]

In [22]:
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    input_length=max_length,
    name="trainable_embedding"
)

In [24]:
# embedding output
#take one paddesreview as input to the embedding layer
sample_input = X_train_padded[:1]
#Pass the sample input through the embedding layer
sample_embedding = embedding_layer(sample_input)
print(sample_embedding.shape)

(1, 200, 128)


### Compare the representations

In [25]:
representation_summary = pd.DataFrame({
    "Representation": [
        "Integer Encoding",
        "TF-IDF",
        "Word2Vec Embedding",
        "Trainable Embedding"
    ],
    "Type": [
        "Sparse integer IDs",
        "Sparse numerical features",
        "Dense word vectors",
        "Dense trainable word vectors"
    ]
})

representation_summary

,Representation,Type
0,Integer Encoding,Sparse integer IDs
1,TF-IDF,Sparse numerical features
2,Word2Vec Embedding,Dense word vectors
3,Trainable Embedding,Dense trainable word vectors


In [26]:
print("Integer Encoding:", X_train_padded.shape)
print("TF-IDF:", X_train_tfidf.shape)
print("Word2Vec dimension:", word2vec_model.wv.vector_size)
print("Trainable Embedding dimension:", embedding_dim)

Integer Encoding: (34705, 200)
TF-IDF: (34705, 20000)
Word2Vec dimension: 100
Trainable Embedding dimension: 128


In [27]:
with open("../models/tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(tfidf, file)

In [28]:
from scipy.sparse import save_npz

save_npz("../models/X_train_tfidf.npz", X_train_tfidf)
save_npz("../models/X_val_tfidf.npz", X_val_tfidf)
save_npz("../models/X_test_tfidf.npz", X_test_tfidf)

In [29]:
word2vec_model.save("../models/word2vec.model")